# Nepali News Classifier - Training Statistics

This notebook loads the trained model and test data, then generates
visualizations to analyze model performance. Each section answers a
specific diagnostic question:

1. **Dataset Distribution** — Are the splits balanced? Class imbalance
   causes the model to bias toward majority classes.
2. **Training Loss Curve** — Did training converge? Is the learning
   rate schedule working? Are we overfitting?
3. **Confusion Matrix** — Which categories does the model confuse?
   Reveals semantic overlap between classes.
4. **Per-Class Metrics** — Precision/recall/F1 per category. Shows
   which classes need more data or better features.
5. **Confidence Distribution** — Is the model well-calibrated? High
   confidence on wrong predictions indicates overfitting.
6. **Text Length Distribution** — Does truncation at MAX_LEN bias
   certain categories? Shorter texts may lose context.

In [ ]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------
# json:             Load labels.json (category -> integer ID mapping)
# pandas:           Read CSV data splits and build metrics DataFrames
# matplotlib:       Plotting library for all visualizations
# numpy:            Array operations for probability analysis
# pathlib:          Cross-platform file paths (works on Windows/Linux/Mac)
# sklearn.metrics:  Confusion matrix and classification report computation
# torch:            Run model inference on GPU/CPU
# transformers:     Load the fine-tuned BERT model and tokenizer

import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from sklearn.metrics import confusion_matrix, classification_report
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

## 1. Dataset Distribution

Load the train/val/test CSVs and the label mapping. We check that:
- All splits have roughly the same class distribution (balanced sampling worked)
- The label IDs in the CSVs match the labels.json mapping
- Split sizes are as expected (80/10/10 ratio)

In [ ]:
# Path to the data directory containing CSVs and labels.json
DATA_DIR = Path("data")

# Load the three data splits as pandas DataFrames.
# Each CSV has columns: text (str), category (str), label (int)
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "val.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

# Load the label mapping: {"economy": 0, "global": 1, ...}
# This is the source of truth for category-to-ID assignments.
# All scripts (train.py, classify_cli.py) use this same file.
with open(DATA_DIR / "labels.json") as f:
    label2id = json.load(f)

# Invert the mapping: {0: "economy", 1: "global", ...}
# Needed for displaying human-readable labels in plots and reports.
id2label = {v: k for k, v in label2id.items()}

# Print summary to verify splits loaded correctly
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"Categories: {list(label2id.keys())}")

In [ ]:
# ---------------------------------------------------------------------------
# Bar charts: sample count per category in each split
# ---------------------------------------------------------------------------
# Why this matters: If one category has 10x more samples than another,
# the model will be biased toward the majority class. Balanced splits
# (from prepare_data.py's sampling step) prevent this.

# Create a 1x3 grid of subplots, one per split (train/val/test)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Iterate over each split and plot its category distribution
for ax, (name, df) in zip(axes, [("Train", train_df), ("Val", val_df), ("Test", test_df)]):
    # Count samples per category, sorted alphabetically for consistent ordering
    counts = df["category"].value_counts().sort_index()

    # Use Set2 colormap for distinct, colorblind-friendly colors
    colors = plt.cm.Set2(np.linspace(0, 1, len(counts)))
    counts.plot.bar(ax=ax, color=colors, edgecolor="black", linewidth=0.5)

    ax.set_title(f"{name} Split ({len(df)} samples)")
    ax.set_ylabel("Count")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45)  # Rotate labels for readability

    # Add count labels above each bar for precise reading
    for i, v in enumerate(counts.values):
        ax.text(i, v + 20, str(v), ha="center", va="bottom", fontsize=9)

# Tight layout prevents label clipping
plt.tight_layout()
plt.show()

## 2. Training Loss Curve

Read the trainer_state.json from the last checkpoint to visualize:
- **Training loss** over steps: should decrease steadily. Plateau = convergence.
  Spikes = unstable gradients (try lowering learning rate).
- **Learning rate** over steps: should ramp up (warmup) then decay linearly.
  The decay lets the model settle into a minimum without oscillating.

In [ ]:
# Path to the model directory containing checkpoints
MODEL_DIR = Path("model")

# Find the latest checkpoint's trainer_state.json
# HuggingFace Trainer saves training logs (loss, LR, eval metrics) here.
# We use the last checkpoint because it contains the full training history.
log_file = None
for ckpt in sorted(MODEL_DIR.glob("checkpoints/checkpoint-*")):
    state_file = ckpt / "trainer_state.json"
    if state_file.exists():
        log_file = state_file

if log_file:
    # Parse the training log history
    with open(log_file) as f:
        state = json.load(f)
    logs = state["log_history"]

    # Separate training logs (have 'loss' key) from eval logs (have 'eval_loss' key)
    # Training logs are recorded every logging_steps (50 steps by default)
    # Eval logs are recorded once per epoch
    train_logs = [l for l in logs if "loss" in l]
    eval_logs = [l for l in logs if "eval_loss" in l]

    # --- Plot 1: Training loss over steps ---
# --- Plot 2: Learning rate schedule ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Extract step numbers, loss values, and learning rates from logs
    steps = [l["step"] for l in train_logs]
    losses = [l["loss"] for l in train_logs]
    lrs = [l["learning_rate"] for l in train_logs]

    # Left plot: Training loss
    # Loss should decrease over time. If it plateaus, the model has converged.
    # If it starts increasing, the model is overfitting (use early stopping).
    ax1.plot(steps, losses, "o-", color="steelblue", linewidth=2, markersize=4)
    ax1.set_xlabel("Step")
    ax1.set_ylabel("Training Loss")
    ax1.set_title("Training Loss Over Steps")
    ax1.grid(True, alpha=0.3)

    # Right plot: Learning rate schedule
    # Shows warmup (ramp up) followed by linear decay to ~0.
    # The warmup prevents large gradient updates at the start when the
    # classification head is randomly initialized and unstable.
    ax2.plot(steps, lrs, "o-", color="coral", linewidth=2, markersize=4)
    ax2.set_xlabel("Step")
    ax2.set_ylabel("Learning Rate")
    ax2.set_title("Learning Rate Schedule")
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Print epoch-1 eval metrics if available
    # This gives a quick sanity check: does the model learn anything in epoch 1?
    if eval_logs:
        print(f"\nEpoch 1 eval accuracy: {eval_logs[0]['eval_accuracy']:.4f}")
        print(f"Epoch 1 eval loss: {eval_logs[0]['eval_loss']:.4f}")
else:
    print("No checkpoint logs found.")

## 3. Run Model Evaluation on Test Set

Load the saved model and run batch inference on the test set. This cell
produces the raw predictions and probability distributions needed for
all subsequent analysis (confusion matrix, per-class metrics, confidence).

We use batched inference (batch_size=32) for GPU efficiency rather than
processing one sample at a time.

In [ ]:
# Max token length for inference. Can be shorter than training's MAX_LEN
# (256) because most classification signal is in the first sentence.
# Using 128 here speeds up inference without significant accuracy loss.
MAX_LEN = 128

# Select GPU if available, otherwise CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Load the fine-tuned model and tokenizer from the checkpoint directory.
# The tokenizer converts raw text -> token IDs that BERT understands.
# The model includes the classification head with id2label mapping.
# AutoTokenizer/AutoModelForSequenceClassification auto-detect the correct
# class from the saved config.json in MODEL_DIR.
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_DIR))

# Set to evaluation mode: disables dropout layers for deterministic output.
# Without this, dropout would randomly zero activations during inference,
# giving slightly different predictions each run.
model.eval()

# Move model to GPU/CPU. All tensors must be on the same device as the model.
model.to(device)

print(f"Model loaded. Classes: {list(model.config.id2label.values())}")

In [ ]:
# ---------------------------------------------------------------------------
# Batched inference on the test set
# ---------------------------------------------------------------------------
# We process texts in batches of 32 for GPU efficiency.
# Each batch is tokenized, moved to device, passed through the model,
# and softmax is applied to convert logits -> probabilities.

texts = test_df["text"].tolist()       # Raw Nepali text strings
true_labels = test_df["label"].tolist() # Ground-truth integer labels

BATCH = 32          # Batch size for inference (matches GPU memory)
pred_labels = []    # Predicted class IDs (argmax of probabilities)
all_probs = []      # Full probability distributions for each sample

# torch.no_grad() disables gradient computation:
# - Reduces memory usage by ~50% (no gradient tensors stored)
# - Speeds up inference by ~2x (no backward pass overhead)
with torch.no_grad():
    for i in range(0, len(texts), BATCH):
        batch_texts = texts[i : i + BATCH]

        # Tokenize: converts text to input_ids + attention_mask tensors
        # padding=True: pad shorter texts to match the longest in the batch
        # truncation=True: cut texts longer than MAX_LEN tokens
        enc = tokenizer(
            batch_texts,
            max_length=MAX_LEN,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )
        # Move all tensors to the same device as the model (GPU/CPU)
        enc = {k: v.to(device) for k, v in enc.items()}

        # Forward pass: model outputs raw logits (unnormalized scores)
        logits = model(**enc).logits

        # Softmax converts logits to probabilities that sum to 1.0
        # dim=-1 applies softmax across the class dimension
        probs = torch.softmax(logits, dim=-1)

        # argmax picks the class with highest probability
        pred_labels.extend(probs.argmax(dim=-1).cpu().tolist())

        # Store full probability distributions for confidence analysis later
        all_probs.extend(probs.cpu().tolist())

print(f"Evaluated {len(texts)} samples.")

## 4. Confusion Matrix

A confusion matrix shows where the model gets confused. Each row is the
true class, each column is the predicted class. The diagonal shows correct
predictions. Off-diagonal values show which classes are confused.

Common patterns:
- High off-diagonal between two classes = semantic overlap (e.g., national/politics)
- One row with many off-diagonal entries = that class is systematically misclassified
- Symmetric off-diagonal confusion = bidirectional confusion (A->B and B->A)

In [ ]:
# Build ordered list of category names matching label IDs 0, 1, 2, ...
label_names = [id2label[i] for i in range(len(id2label))]

# Compute confusion matrix: cm[i][j] = number of samples with true label i
# predicted as label j. The diagonal cm[i][i] = correct predictions for class i.
cm = confusion_matrix(true_labels, pred_labels)

# Plot as a heatmap
fig, ax = plt.subplots(figsize=(8, 6))

# Blues colormap: darker = more samples. 'nearest' interpolation keeps cells sharp.
im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)

# Label axes with category names
ax.set(
    xticks=np.arange(len(label_names)),
    yticks=np.arange(len(label_names)),
    xticklabels=label_names,
    yticklabels=label_names,
    ylabel="True Label",
    xlabel="Predicted Label",
    title="Confusion Matrix",
)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

# Add count labels inside each cell
# Use white text on dark cells (high count), black on light cells (low count)
thresh = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(
            j, i, format(cm[i, j], "d"),
            ha="center", va="center",
            color="white" if cm[i, j] > thresh else "black",
        )

plt.tight_layout()
plt.show()

## 5. Per-Class Metrics

Precision, recall, and F1-score per category:

- **Precision** = TP / (TP + FP): Of all samples predicted as this class,
  how many actually are? High precision = few false positives.
- **Recall** = TP / (TP + FN): Of all actual samples of this class,
  how many did we find? High recall = few false negatives.
- **F1-score** = harmonic mean of precision and recall: balances both.

Low precision on a class means the model over-predicts it (false positives).
Low recall means the model under-predicts it (false negatives).

In [ ]:
# Generate a full classification report with per-class metrics.
# output_dict=True returns a dictionary instead of printing, so we can
# extract values programmatically for plotting.
report = classification_report(true_labels, pred_labels, target_names=label_names, output_dict=True)

# Build a DataFrame for easy display and plotting
metrics_df = pd.DataFrame({
    "precision": [report[c]["precision"] for c in label_names],
    "recall": [report[c]["recall"] for c in label_names],
    "f1-score": [report[c]["f1-score"] for c in label_names],
    "support": [int(report[c]["support"]) for c in label_names],  # Number of true samples
}, index=label_names)

# Print the metrics table
print(metrics_df.to_string())
print(f"\nAccuracy: {report['accuracy']:.4f}")

In [ ]:
# ---------------------------------------------------------------------------
# Grouped bar chart: Precision, Recall, F1 per category
# ---------------------------------------------------------------------------
# This visualization makes it easy to compare metrics across categories
# and identify which classes need improvement.

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(label_names))  # X positions for category groups
width = 0.25  # Width of each bar (3 bars per group)

# Plot three bars per category: precision (blue), recall (red), F1 (green)
ax.bar(x - width, metrics_df["precision"], width, label="Precision", color="steelblue")
ax.bar(x, metrics_df["recall"], width, label="Recall", color="coral")
ax.bar(x + width, metrics_df["f1-score"], width, label="F1-Score", color="seagreen")

ax.set_ylabel("Score")
ax.set_title("Per-Class Precision, Recall, F1")
ax.set_xticks(x)
ax.set_xticklabels(label_names, rotation=45, ha="right")
ax.set_ylim(0, 1.05)  # Leave room for text labels above bars
ax.legend()
ax.grid(True, alpha=0.3, axis="y")  # Horizontal gridlines for readability

# Add numeric labels above each bar for precise reading
for i, (p, r, f) in enumerate(zip(metrics_df["precision"], metrics_df["recall"], metrics_df["f1-score"])):
    ax.text(i - width, p + 0.02, f"{p:.2f}", ha="center", va="bottom", fontsize=8)
    ax.text(i, r + 0.02, f"{r:.2f}", ha="center", va="bottom", fontsize=8)
    ax.text(i + width, f + 0.02, f"{f:.2f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.show()

## 6. Confidence Distribution

Two diagnostic plots for model calibration:

1. **Histogram of max probabilities** — Separated by correct/wrong predictions.
   A well-calibrated model shows:
   - Correct predictions clustered near 1.0 (high confidence)
   - Wrong predictions clustered near 0.5 (uncertain, not confidently wrong)
   If wrong predictions have high confidence, the model is overconfident
   and likely overfitting.

2. **Accuracy by confidence bucket** — Shows actual accuracy at each confidence
   level. Well-calibrated means: samples the model is 90% confident about
   should be correct ~90% of the time. If accuracy at 90% confidence is
   only 60%, the model is poorly calibrated.

In [ ]:
# Convert to numpy for easier slicing
all_probs = np.array(all_probs)

# For each sample, take the maximum probability across all classes.
# This is the model's "confidence" in its top prediction.
max_probs = all_probs.max(axis=1)

# Boolean mask: True where prediction matches ground truth
correct = np.array(pred_labels) == np.array(true_labels)

# --- Plot 1: Confidence histograms (correct vs wrong) ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Overlapping histograms: green for correct, red for wrong predictions
# If red bars extend to the right (high confidence), the model is
# confidently wrong — a sign of overfitting or label noise.
ax1.hist(max_probs[correct], bins=30, alpha=0.7, label="Correct", color="seagreen", edgecolor="black")
ax1.hist(max_probs[~correct], bins=30, alpha=0.7, label="Wrong", color="coral", edgecolor="black")
ax1.set_xlabel("Confidence (max probability)")
ax1.set_ylabel("Count")
ax1.set_title("Confidence Distribution: Correct vs Wrong")
ax1.legend()
ax1.grid(True, alpha=0.3)

# --- Plot 2: Accuracy by confidence level ---
# Bin predictions into 5% confidence buckets (0-5%, 5-10%, ..., 95-100%)
# and compute accuracy within each bucket.
bins = np.arange(0, 1.05, 0.05)
acc_per_bin = []
counts_per_bin = []
bin_centers = []
for lo, hi in zip(bins[:-1], bins[1:]):
    # Select samples whose max probability falls in [lo, hi)
    mask = (max_probs >= lo) & (max_probs < hi)
    if mask.sum() > 0:
        acc_per_bin.append(correct[mask].mean())  # Accuracy in this bucket
        counts_per_bin.append(mask.sum())           # How many samples
        bin_centers.append((lo + hi) / 2)           # Center of bucket

# Bar chart: each bar shows accuracy at a confidence level
# For a well-calibrated model, the bar height should roughly match
# the x-axis label (e.g., the 80% bucket should have ~80% accuracy)
ax2.bar([f"{c:.0%}" for c in bin_centers], acc_per_bin, color="steelblue", edgecolor="black")
ax2.set_xlabel("Confidence Bucket")
ax2.set_ylabel("Accuracy")
ax2.set_title("Accuracy by Confidence Level")
ax2.set_ylim(0, 1.05)
ax2.grid(True, alpha=0.3, axis="y")
plt.setp(ax2.get_xticklabels(), rotation=45, ha="right")

plt.tight_layout()
plt.show()

## 7. Text Length Distribution

Show how long the training texts are (in characters) per category.

This matters because:
- Texts longer than MAX_LEN tokens get truncated, losing context
- If one category consistently has longer texts, truncation hurts it more
- Short texts (< 50 chars) may lack enough signal for classification

In [ ]:
# Compute character count for each training text
train_df["text_len"] = train_df["text"].str.len()

# Plot overlapping histograms, one per category
# Different colors and alpha (transparency) let us see the overlap
fig, ax = plt.subplots(figsize=(10, 5))
for cat in sorted(train_df["category"].unique()):
    subset = train_df[train_df["category"] == cat]["text_len"]
    ax.hist(subset, bins=40, alpha=0.5, label=cat, edgecolor="black", linewidth=0.3)

ax.set_xlabel("Character Count")
ax.set_ylabel("Count")
ax.set_title("Text Length Distribution by Category (Train)")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()